In [3]:
import requests
import config
import pandas as pd
from sqlalchemy import create_engine

In [2]:
# Replace with your actual client ID, token, and the channel username
CLIENT_ID = 'cli_fb70343863c4fb0936e00e99'
TOKEN = '7ee798864a8323934777d56cdb9a42e8fb5175d784b3bea2f363b76ae9b6fa8fc4a44e24e1a2b9065224fb0c9df56207063d8b5960bd2cc63605a2e500110db2'
CHANNEL_USERNAME = 'Vaush'

# Define the endpoint URL
url = f'https://matrix.sbapis.com/b/youtube/statistics?query={CHANNEL_USERNAME}'

# Set up headers with your client ID and token
# Add the history parameter in the headers
headers = {
    'ClientID': CLIENT_ID,
    'Token': TOKEN,
    'History': 'archive'  # Request up to 3 years of data
}
response = requests.get(f'https://matrix.sbapis.com/b/youtube/statistics?query={CHANNEL_USERNAME}', headers=headers)

# Make the request to the Social Blade API
response = requests.get(url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    data = response.json()
    # Extract the 'daily' data which contains historical statistics
    daily_data = data.get('data', {}).get('daily', [])
    # Convert the data into a pandas DataFrame
    df = pd.DataFrame(daily_data)
    # Convert the 'date' column to datetime format
    df['date'] = pd.to_datetime(df['date'])
    # Sort the DataFrame by date
    df = df.sort_values(by='date')
    # Display the DataFrame
    print(df)
else:
    print(f'Failed to retrieve data: {response.status_code}')


           date    subs      views
1094 2021-12-09  384000  164123784
1093 2021-12-10  384000  164336081
1092 2021-12-11  384000  164611674
1091 2021-12-12  384000  164779457
1090 2021-12-13  384000  164984670
...         ...     ...        ...
4    2024-12-03  490000  385170513
3    2024-12-04  490000  385170513
2    2024-12-05  490000  385497721
1    2024-12-06  491000  386112760
0    2024-12-07  491000  386112760

[1095 rows x 3 columns]


In [5]:
# Example connection details, replace with your actual credentials
DATABASE_TYPE = config.DATABASE_TYPE
DBAPI = config.DBAPI
ENDPOINT = config.ENDPOINT
USER = config.USER
PASSWORD = config.PASSWORD
PORT = config.PORT
DATABASE = config.YOUTUBE_DATABASE

# Create the database URL
DATABASE_URL = f'{DATABASE_TYPE}+{DBAPI}://{USER}:{PASSWORD}@{ENDPOINT}:{PORT}/{DATABASE}'

# Create the engine
engine = create_engine(DATABASE_URL)

# Make all column names uppercase
df.columns = df.columns.str.upper()

# Insert DataFrame into PostgreSQL table
df.to_sql('SOCIALBLADE_STATISTICS', engine, if_exists='append', index=False)

95